In [1]:
import pandas as pd
import numpy as np

In [9]:
# --- Configuration based on your Calliope model ---
START_DATE = '2023-01-01 00:00'
END_DATE = '2023-12-31 23:00'
FILENAME = '../timeseries_data/biomass_avail_seasonal.csv' # Adjust path as needed

# Define the unavailability period (Summer/Shoulder months)
SUMMER_START = '05-01'  # May 1st
SUMMER_END = '09-14'    # September 14th


In [3]:
# --- 1. Create the full annual hourly index ---
index = pd.date_range(start=START_DATE, end=END_DATE, freq='H')
index

C:\Users\dnouicer.THDITZ0329\AppData\Local\Temp\ipykernel_4592\1879216351.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  index = pd.date_range(start=START_DATE, end=END_DATE, freq='H')


DatetimeIndex(['2023-01-01 00:00:00', '2023-01-01 01:00:00',
               '2023-01-01 02:00:00', '2023-01-01 03:00:00',
               '2023-01-01 04:00:00', '2023-01-01 05:00:00',
               '2023-01-01 06:00:00', '2023-01-01 07:00:00',
               '2023-01-01 08:00:00', '2023-01-01 09:00:00',
               ...
               '2023-12-31 14:00:00', '2023-12-31 15:00:00',
               '2023-12-31 16:00:00', '2023-12-31 17:00:00',
               '2023-12-31 18:00:00', '2023-12-31 19:00:00',
               '2023-12-31 20:00:00', '2023-12-31 21:00:00',
               '2023-12-31 22:00:00', '2023-12-31 23:00:00'],
              dtype='datetime64[ns]', length=8760, freq='h')

In [4]:
# --- 2. Initialize the availability data frame ---
# The column name MUST match the technology name used in locations.yaml (biomass)
df = pd.DataFrame(index=index)
df['biomass'] = 1.0  # Start by assuming the plant is available (1.0) all year
df

,biomass
2023-01-01 00:00:00,1.0
2023-01-01 01:00:00,1.0
2023-01-01 02:00:00,1.0
2023-01-01 03:00:00,1.0
2023-01-01 04:00:00,1.0
...,...
2023-12-31 19:00:00,1.0
2023-12-31 20:00:00,1.0
2023-12-31 21:00:00,1.0
2023-12-31 22:00:00,1.0


In [5]:
# --- 3. Apply the seasonal constraint (set to 0.0 in summer) ---
# Check if the month is outside the heating season range (mid-Sept to early May)
summer_condition = (
    # All days from May 1st up to and including September 14th
    (df.index.strftime('%m-%d') >= SUMMER_START) & 
    (df.index.strftime('%m-%d') <= SUMMER_END)
)

# Apply the 0.0 availability factor to the summer period
df.loc[summer_condition, 'biomass'] = 0.0
df

,biomass
2023-01-01 00:00:00,1.0
2023-01-01 01:00:00,1.0
2023-01-01 02:00:00,1.0
2023-01-01 03:00:00,1.0
2023-01-01 04:00:00,1.0
...,...
2023-12-31 19:00:00,1.0
2023-12-31 20:00:00,1.0
2023-12-31 21:00:00,1.0
2023-12-31 22:00:00,1.0


In [10]:
# --- 4. Save the file ---
# Calliope prefers the index to be saved as the first column without a header, 
# and often requires no quoting.
df.to_csv(FILENAME, header=True, index=True, date_format='%Y-%m-%d %H:%M:%S')

print(f"✅ Generated {FILENAME} successfully!")
print("---")
print("First few rows:")
print(df.head())
print("---")
print("Summer shutdown period preview:")
print(df.loc['2023-04-30 22:00':'2023-05-01 01:00'])
print("---")
print("Autumn startup period preview:")
print(df.loc['2023-09-14 22:00':'2023-09-15 01:00'])

✅ Generated ../timeseries_data/biomass_avail_seasonal.csv successfully!
---
First few rows:
                     biomass
2023-01-01 00:00:00      1.0
2023-01-01 01:00:00      1.0
2023-01-01 02:00:00      1.0
2023-01-01 03:00:00      1.0
2023-01-01 04:00:00      1.0
---
Summer shutdown period preview:
                     biomass
2023-04-30 22:00:00      1.0
2023-04-30 23:00:00      1.0
2023-05-01 00:00:00      0.0
2023-05-01 01:00:00      0.0
---
Autumn startup period preview:
                     biomass
2023-09-14 22:00:00      0.0
2023-09-14 23:00:00      0.0
2023-09-15 00:00:00      1.0
2023-09-15 01:00:00      1.0
